In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import os 
import glob

In [2]:
adata = sc.read_h5ad('/data1st1/junyi/correctdata/GSE118767/combined_mix.h5ad')

In [3]:
adata

AnnData object with n_obs × n_vars = 12711 × 41442
    obs: 'protocol', 'title', 'source name', 'organism', 'barcode', 'barcode_dropseq', 'n_reads_dropseq', 'source_file', 'sample_id'
    layers: None (.X)

In [4]:
# ==== 把 CellBench_metadata 里的全部 metadata 导入 pandas（GSE118767 的 ground truth 标签）====
meta_dir = "/data1st1/junyi/correctdata/GSE118767/CellBench_metadata/"

# 1) 读入全部 14 个 metadata 文件，行名 = 细胞/孔位，加 dataset 列标注来源
meta_files = sorted(glob.glob(os.path.join(meta_dir, "*.metadata.csv.gz")))
print(f"找到 {len(meta_files)} 个 metadata 文件")

frames = []
for fp in meta_files:
    dataset = os.path.basename(fp).replace(".metadata.csv.gz", "")
    df = pd.read_csv(fp, index_col=0)
    df.index.name = "cell"
    df["dataset"] = dataset
    frames.append(df)
meta_all = pd.concat(frames)
print("合并后形状:", meta_all.shape)

# 2) 修正 CellBench 的数据 bug：5cl 板的 p2/p3 行名被误标成 p1_ 前缀 -> 统一剥掉 p1_
mask5cl = meta_all["dataset"].str.startswith("sc_celseq2_5cl")
meta_all.index = [i[3:] if (m and i.startswith("p1_")) else i
                  for m, i in zip(mask5cl, meta_all.index)]

# 3) 只保留标签列（去掉 unaligned/mapped_* 等纯 QC 统计列），方便使用
LABEL_COLS = ["H1975", "H2228", "HCC827", "traj", "poor_quality",          # cellmix: 每孔各细胞系细胞数
              "H2228_prop", "H1975_prop", "HCC827_prop", "mRNA_amount", "mix",  # RNAmix: RNA 混合比例
              "cell_line", "cell_line_demuxlet", "demuxlet_cls"]           # 单细胞: 每细胞细胞系身份
labels = meta_all[LABEL_COLS + ["dataset"]].copy()
print()
print("=== 每个数据集的标签覆盖（非空行数）===")
print(labels.groupby("dataset")[LABEL_COLS].apply(lambda s: s.notna().sum()).to_string())
print()
print("=== 抽查 ===")
labels[labels["dataset"] == "cellmix1"].head(3)

找到 14 个 metadata 文件
合并后形状: (7992, 47)

=== 每个数据集的标签覆盖（非空行数）===
                   H1975  H2228  HCC827  traj  poor_quality  H2228_prop  H1975_prop  HCC827_prop  mRNA_amount  mix  cell_line  cell_line_demuxlet  demuxlet_cls
dataset                                                                                                                                                        
RNAmix_celseq2         0      0       0     0             0         340         340          340          340  340          0                   0             0
RNAmix_sortseq         0      0       0     0             0         296         296          296          296  296          0                   0             0
cellmix1             266    266     266   266           266           0           0            0            0    0          0                   0             0
cellmix2             268    268     268   268           268           0           0            0            0    0          0            

,H1975,H2228,HCC827,traj,poor_quality,H2228_prop,H1975_prop,HCC827_prop,mRNA_amount,mix,cell_line,cell_line_demuxlet,demuxlet_cls,dataset
L19,7.0,0.0,2.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix1
A10,0.0,0.0,9.0,YES,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix1
A11,0.0,0.0,9.0,YES,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix1


In [7]:
labels.drop_duplicates(subset=["dataset"])

,H1975,H2228,HCC827,traj,poor_quality,H2228_prop,H1975_prop,HCC827_prop,mRNA_amount,mix,cell_line,cell_line_demuxlet,demuxlet_cls,dataset
L19,NaN,NaN,NaN,NaN,NaN,0.68,0.16,0.16,30.0,6.0,NaN,NaN,NaN,RNAmix_celseq2
L19,NaN,NaN,NaN,NaN,NaN,0.68,0.16,0.16,30.0,6.0,NaN,NaN,NaN,RNAmix_sortseq
L19,7.0,0.0,2.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix1
L19,7.0,0.0,2.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix2
L19,7.0,0.0,2.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix3
L19,7.0,0.0,2.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix4
O1,10.0,10.0,70.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cellmix5
CELL_000001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HCC827,HCC827,SNG,sc_10x
Lib90_00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HCC827,HCC827,SNG,sc_10x_5cl
A1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,H1975,H1975,SNG,sc_celseq2


In [9]:
adata.obs.drop_duplicates(subset=["protocol"])

,protocol,title,source name,organism,barcode,barcode_dropseq,n_reads_dropseq,source_file,sample_id
POP_O1,POP,NaN,NaN,NaN,NaN,NaN,NaN,GSM3295023_POP_counts.csv,GSM3295023_POP_counts.csv
Mix1_A3,SC1,NaN,NaN,NaN,NaN,NaN,NaN,GSM3295024_SC1_counts.csv,GSM3295024_SC1_counts.csv
Mix2_A3,SC2,NaN,NaN,NaN,NaN,NaN,NaN,GSM3295025_SC2_counts.csv,GSM3295025_SC2_counts.csv
Mix3_A3,SC3,NaN,NaN,NaN,NaN,NaN,NaN,GSM3295026_SC3_counts.csv,GSM3295026_SC3_counts.csv
Mix4_A3,SC4,NaN,NaN,NaN,NaN,NaN,NaN,GSM3295027_SC4_counts.csv,GSM3295027_SC4_counts.csv
CelSeq2_Mixture_Sample_A1,CelSeq2_Mixture_Sample,NaN,NaN,NaN,NaN,NaN,NaN,GSM3305230_CelSeq2_Mixture_Sample_counts.csv,GSM3305230_CelSeq2_Mixture_Sample_counts.csv
SortSeq_Mixture_Sample_A1,SortSeq_Mixture_Sample,NaN,NaN,NaN,NaN,NaN,NaN,GSM3305231_SortSeq_Mixture_Sample_counts.csv,GSM3305231_SortSeq_Mixture_Sample_counts.csv
A1,NN84_Celseq2_SC,CelSeq2_SC_Sample_A1,lung adenocarcinoma,Homo sapiens,GTAGCTCA,NaN,NaN,GSM3336845_NN84_Celseq2_SC_counts.csv,GSM3336845_NN84_Celseq2_SC_counts.csv
CELL_000001,dropseq,NaN,NaN,NaN,NaN,ACAAAGATTCTA,425865.0,GSM3336849_dropseq_gene_count.csv,GSM3336849_dropseq_gene_count.csv
Lib90_00000,10x,NaN,NaN,NaN,NaN,NaN,NaN,GSM3618014_gene_count.csv,GSM3618014_gene_count.csv
